# PanTS SegResNet — local RTX 4070 production run

Orchestration and teaching only. Model, loss, transforms, trainer, evaluator and
checkpointing live in `src/` and are never redefined here.

## 0. Experiment identity

The two arms use the same scientific training protocol, with **initialization as
the intended intervention**:

| | Random | SuPreM |
|---|---|---|
| initialization | random | 81 of 83 tensors from supervised abdominal-CT pretraining |
| training hardware | Colab Tesla T4 | RTX 4070 Laptop (this machine) |
| fold / epochs / batch / samples / accum | 0 / 64 / 2 / 2 / 1 | identical |
| lr / wd / seed / workers | 1e-4 / 1e-5 / 317 / 4 | identical |
| preprocessing | RAS, 1.5 mm, HU [-175,250] → [0,1] | identical |
| sampler | other:pancreas:lesion = 1:2:3 | identical |

Training hardware and software differ between the arms and are recorded per run,
so this is **not** a hardware-controlled ablation.

**PanTS-te (IDs > 9000) is locked.** Nothing in this notebook may read it.

## 1. Hardware and software

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.used,memory.free,temperature.gpu --format=csv
!nproc; free -h | head -2; swapon --show; df -h . | tail -1

In [ ]:
import platform, sys, torch, monai, numpy as np

print(f"python {platform.python_version()}  torch {torch.__version__}  monai {monai.__version__}  numpy {np.__version__}")
print(f"cuda available {torch.cuda.is_available()}  runtime {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}  addressable {p.total_memory / 2**30:.3f} GiB  SMs {p.multi_processor_count}")
assert "pants_sabin" in sys.executable, f"wrong kernel: {sys.executable}"

`nvidia-smi` reports 8188 MiB while CUDA sees 7806 MiB. The 382 MiB difference is
`memory.reserved`, the driver's own allocation. **7806 MiB is the real ceiling.**

Thermal limits are reported as *margins*, not absolutes: watch that `T.Limit` stays
positive and that `clocks_event_reasons.active` shows no thermal reason.

## 2. Paths

The only cell to edit on a different machine. Defaults come from the environment.

In [ ]:
import os
from pathlib import Path

PROJECT      = Path(os.environ.get("PANTS_PROJECT", Path.cwd().parent)).resolve()
PREPARED     = Path(os.environ.get("PANTS_PREPARED_ROOT", PROJECT.parent / "PanTS_prepared/segresnet"))
SUPREM       = Path(os.environ.get("SUPREM_CHECKPOINT", PROJECT.parent / "pretrained/supervised_suprem_segresnet_2100.pth"))
SPLIT        = PROJECT / "pants_cv_v1.json"
MANIFEST     = PREPARED / "manifest.json"
OUTPUT_ROOT  = PROJECT / "PanTS_run"          # gitignored

os.chdir(PROJECT)
for name, path in [("project", PROJECT), ("prepared", PREPARED), ("split", SPLIT),
                   ("manifest", MANIFEST), ("suprem", SUPREM), ("output root", OUTPUT_ROOT)]:
    print(f"{name:<12} {path}   {'OK' if path.exists() else 'MISSING'}")

## 3. Prepared-cache verification

Fast structural checks only. The one-time raw-NIfTI rederivation audit (6/6
bit-identical, including degenerate-affine cases) is recorded in
`METHODS_AND_QC.md` and is deliberately **not** repeated at startup.

In [ ]:
import hashlib, json

def content_sha256(payload) -> str:
    """The trainer's canonical hash, so this is comparable to checkpoint provenance."""
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()

ids = sorted(int(p.name.removesuffix(".npz").split("_")[1]) for p in (PREPARED / "cases").glob("*.npz"))
assert len(ids) == 9000, f"expected 9000 cases, found {len(ids)}"
assert ids == list(range(1, 9001)), "case IDs are not exactly 1..9000"
assert not any(i > 9000 for i in ids), "PanTS-te leaked into the cache"

manifest = json.loads(MANIFEST.read_text())
split = json.loads(SPLIT.read_text())
print("cases          ", len(ids), "| ids 1..9000 exact | no PanTS-te")
print("manifest sha   ", content_sha256(manifest))
print("split sha      ", content_sha256(split))
print("preprocessing  ", json.loads((PREPARED / 'preprocessing.json').read_text()))

Those two hashes must equal the ones inside the Random checkpoint
(`8137b8b6…`, `0a3610ff…`), or the two arms did not see the same data.

## 4. One prepared case

In [ ]:
import numpy as np

with np.load(PREPARED / "cases" / "PanTS_00000003.npz", allow_pickle=False) as archive:
    image, label = archive["image"], archive["label"]

print(f"image {image.shape} {image.dtype}  range [{image.min():.3f}, {image.max():.3f}]")
print(f"label {label.shape} {label.dtype}  classes {sorted(np.unique(label))}")
print(f"physical extent {tuple(round(n * 1.5) for n in image.shape)} mm")

Axes are `[D, H, W]` in RAS at 1.5 mm isotropic. Intensity is already the clipped
and rescaled HU window, stored as float16; the label is uint8 with values 0…28.
The channel axis MONAI adds is dropped in storage and restored on load.

## 5. SuPreM checkpoint — hash gate

In [ ]:
EXPECTED = "2db81dc05cd9ea7234ca75e921e53e32b8716dc4cba88a6710742bfc282589a3"

if not SUPREM.exists():
    raise SystemExit(
        f"Download it first:\n  wget -O {SUPREM} "
        "'https://huggingface.co/MrGiovanni/SuPreM/resolve/main/"
        "supervised_suprem_segresnet_2100.pth?download=true'")

digest = hashlib.sha256()
with open(SUPREM, "rb") as handle:
    for chunk in iter(lambda: handle.read(1 << 20), b""):
        digest.update(chunk)
assert digest.hexdigest() == EXPECTED, "SHA256 MISMATCH - HARD STOP"
print(f"{SUPREM.stat().st_size} bytes  {digest.hexdigest()}  VERIFIED")

## 6. Transfer verification — 81 of 83

In [ ]:
from src.models.segresnet import build_segresnet, suprem_transfer_report, format_transfer_report

model = build_segresnet("random")
report = suprem_transfer_report(model, str(SUPREM))
print(format_transfer_report(report))

assert len(model.state_dict()) == 83
assert len(report["transferable"]) == 81
assert len(report["excluded_output_head"]) == 2
assert not report.get("missing") and not report.get("shape_mismatch")
del model

A `state_dict` is an ordered mapping from parameter name to tensor — the learned
numbers, with no architecture attached.

The two excluded tensors are `conv_final.2.conv.weight` `(29,16,1,1,1)` and
`conv_final.2.conv.bias` `(29,)`: the 1×1×1 convolution mapping 16 decoder features
onto class logits. SuPreM's head predicts a different ontology with a different
channel count, so its rows name structures that are not our classes. Copying them
is not merely shape-invalid, it is semantically meaningless.

Everything else — encoder, decoder, every normalization — transfers. All 83 tensors
are then **trainable**: this is fine-tuning, not frozen-feature extraction.

## 7. What a "batch" is here

| Term | Meaning | Production value |
|---|---|---|
| case | one patient CT volume | ~6.8 M voxels |
| patch | one 96³ crop | 884,736 voxels |
| DataLoader batch | `batch_size` cases | 2 cases |
| network patch batch | `batch_size × samples_per_case` | **4 patches** |
| iteration | one loop body: forward, loss, backward | 3,600 per epoch |
| global step | monotone iteration counter | 230,400 total |
| optimizer update | one `scaler.step()` | = iteration when accumulation=1 |
| epoch | one pass over 7,199 fold-0 training cases | 14,398 patch presentations |

Network input is `[4, 1, 96, 96, 96]`, output `[4, 29, 96, 96, 96]`.
**The whole CT is never a training batch** — training sees stochastic 96³ crops.

## 8. Steady-state memory qualification

Two iterations, not one: AdamW allocates `exp_avg` and `exp_avg_sq` lazily inside
the **first** `optimizer.step()`, so a forward+backward reading understates steady
state by two parameter-sized fp32 buffers. Peak stats are reset between them.

Ephemeral by construction — `output_root` points at a scratch directory and `fit()`
is never called, so no `best.pt`, `latest.pt` or `history.json` can appear.

In [ ]:
import tempfile
from src.training.trainer import SegResNetTrainer, TrainingConfig

scratch = tempfile.mkdtemp(prefix="memqual_")
config = TrainingConfig(
    experiment="MEMQUAL_SCRATCH", initialization="suprem",
    pretrained_checkpoint=str(SUPREM), prepared_root=str(PREPARED),
    fold=0, epochs=64, batch_size=2, samples_per_case=2,
    gradient_accumulation_steps=1, learning_rate=1e-4, weight_decay=1e-5,
    num_workers=4, seed=317, amp=True, device="cuda", output_root=scratch,
    validate_every_epochs=5, monitoring_negatives=50,
)
trainer = SegResNetTrainer(config, manifest=manifest, split=split)
loader = iter(trainer.train_loader)

for tag in ("warmup", "steady_state"):
    batch = next(loader)
    if tag == "warmup":
        print(f"image {tuple(batch['image'].shape)} {batch['image'].dtype}")
        print(f"label {tuple(batch['label'].shape)} {batch['label'].dtype}")
    else:
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    trainer.optimizer.zero_grad(set_to_none=True)
    loss = trainer._forward_loss(batch)
    assert torch.isfinite(loss), "non-finite loss"
    trainer.scaler.scale(loss).backward()
    trainer.scaler.step(trainer.optimizer); trainer.scaler.update()
    torch.cuda.synchronize()
    print(f"{tag:<13} loss {float(loss):.4f}  "
          f"peak alloc {torch.cuda.max_memory_allocated()/2**30:.3f} GiB  "
          f"peak reserved {torch.cuda.max_memory_reserved()/2**30:.3f} GiB")

del trainer, loader, loss, batch
torch.cuda.empty_cache()

**Decision rule.** If `batch_size=2, samples_per_case=2, accumulation=1` fits, use
it — it is the configuration Random ran. Do **not** raise the batch because memory
remains. If it OOMs on an otherwise idle GPU, stop and discuss the
`batch_size=1, samples_per_case=2, accumulation=2` fallback, which restores ~4
patches per update but is **not** mathematically identical: it changes how patches
are grouped before gradients are averaged.

## 9. The loss — teaching only, on detached logits

Production optimizes the combined `DiceCELoss` and logs only that, exactly as the
Random arm did. The decomposition below is never persisted as a production metric:
Random has no matching history, so a SuPreM-only column could not enter the
comparison.

In [ ]:
from monai.losses import DiceCELoss

criterion = DiceCELoss(include_background=False, to_onehot_y=True, softmax=True,
                       lambda_dice=1.0, lambda_ce=1.0)
logits = torch.randn(2, 29, 32, 32, 32)
target = torch.randint(0, 29, (2, 1, 32, 32, 32)).float()

with torch.no_grad():
    dice, ce = criterion.dice(logits, target), criterion.ce(logits, target)
    total = criterion(logits, target)
print(f"dice {float(dice):.6f} + ce {float(ce):.6f} = {float(dice + ce):.6f}")
print(f"DiceCELoss total                      = {float(total):.6f}")
assert torch.allclose(dice + ce, total, atol=1e-6)

For voxel $i$ and class $c$, softmax gives $p_{ic} = e^{z_{ic}} / \sum_k e^{z_{ik}}$,
appropriate because the 29 classes are mutually exclusive.

$$L_{CE} = -\frac{1}{N}\sum_i \log p_{i, y_i}, \qquad
\mathrm{Dice}_c = \frac{2\sum_i p_{ic} y_{ic}}{\sum_i p_{ic} + \sum_i y_{ic} + \epsilon},
\qquad L_{Dice} = 1 - \frac{1}{28}\sum_{c=1}^{28} \mathrm{Dice}_c$$

$$L_{total} = L_{Dice} + L_{CE}$$

Background is excluded from Dice because it dominates the volume and would drive
the mean toward 1 regardless of the organs. Cross-entropy still sees background at
every voxel, so "predict nothing" is still penalized. At initialization
$L_{CE} \approx \ln 29 = 3.367$ and $L_{Dice} \approx 1$.

## 10. Production preflight

In [ ]:
import subprocess

run_dir = OUTPUT_ROOT / "segresnet_suprem"
gates = {
    "working tree clean": subprocess.check_output(["git", "status", "--porcelain"], text=True) == "",
    "prepared cache 9000": len(list((PREPARED / "cases").glob("*.npz"))) == 9000,
    "no PanTS-te": not any(int(p.name.removesuffix(".npz").split("_")[1]) > 9000
                           for p in (PREPARED / "cases").glob("*.npz")),
    "SuPreM verified": SUPREM.exists(),
    "output dir free": not run_dir.exists(),
    "GPU present": torch.cuda.is_available(),
    "disk > 50 GB": os.statvfs(PROJECT).f_bavail * os.statvfs(PROJECT).f_frsize > 50 * 2**30,
}
for name, ok in gates.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")
print("\ncommit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
assert all(gates.values()), "a gate failed - do not launch"

## 11. Production command

**Do not run a 30-hour job inside a notebook kernel.** Print it, then launch it
from a detached shell session.

In [ ]:
print(f"""tmux new -s pants-suprem

cd {PROJECT}
source ~/anaconda3/etc/profile.d/conda.sh && conda activate pants_sabin
set -o pipefail
python -u scripts/train_segresnet.py \\
  --initialization suprem \\
  --experiment segresnet_suprem \\
  --pretrained-checkpoint {SUPREM} \\
  --prepared-root {PREPARED} \\
  --manifest {MANIFEST} \\
  --split {SPLIT} \\
  --fold 0 --epochs 64 \\
  --batch-size 2 --samples-per-case 2 --accumulation 1 \\
  --learning-rate 1e-4 --weight-decay 1e-5 \\
  --num-workers 4 --seed 317 \\
  --save-every-epochs 1 --validate-every-epochs 5 --monitoring-negatives 50 \\
  --output-root {OUTPUT_ROOT} 2>&1 | tee {OUTPUT_ROOT}/segresnet_suprem/train.log

Detach: Ctrl-b then d      Reattach: tmux attach -t pants-suprem""")

No `--limit-cases`, no `--max-steps-per-epoch`, no `--no-amp`. No
`--persistent-output-root`: unlike Colab `/content`, this disk is already durable.

## 12. Resume command

`--epochs 64` is **not** "remaining epochs". The cosine scheduler stored in the
checkpoint was built with `T_max=64`; any other value replays the original curve
against a different horizon.

In [ ]:
print(f"""python -u scripts/train_segresnet.py \\
  --initialization suprem --experiment segresnet_suprem \\
  --pretrained-checkpoint {SUPREM} \\
  --resume {OUTPUT_ROOT}/segresnet_suprem/latest.pt \\
  --prepared-root {PREPARED} --manifest {MANIFEST} --split {SPLIT} \\
  --fold 0 --epochs 64 \\
  --batch-size 2 --samples-per-case 2 --accumulation 1 \\
  --learning-rate 1e-4 --weight-decay 1e-5 \\
  --num-workers 4 --seed 317 \\
  --save-every-epochs 1 --validate-every-epochs 5 --monitoring-negatives 50 \\
  --output-root {OUTPUT_ROOT} 2>&1 | tee -a {OUTPUT_ROOT}/segresnet_suprem/train.log""")

Resume restores model, optimizer, scheduler, GradScaler, RNG, epoch, global step,
best metric, selection epoch **and** the prior `history.json`, so the record stays
one chronological sequence from epoch 0. It refuses to start if the monitoring
subset changed or if the history is not contiguous up to the checkpoint's epoch.

## 13. Monitoring

In [ ]:
print(f"""# progress
tail -f {OUTPUT_ROOT}/segresnet_suprem/train.log

# GPU, once a second
nvidia-smi --query-gpu=timestamp,temperature.gpu,utilization.gpu,memory.used,power.draw \\
  --format=csv -l 1

# what has been written
ls -la {OUTPUT_ROOT}/segresnet_suprem/
python -c "import json;h=json.load(open('{OUTPUT_ROOT}/segresnet_suprem/history.json'));\\
print(len(h),'epochs; last:',h[-1])"

# selection history only
python -c "import json;[print(r['epoch'], r['selection']['mean_dice_on_positive_cases']) \\
  for r in json.load(open('{OUTPUT_ROOT}/segresnet_suprem/history.json')) if 'selection' in r]""")

Monitoring epochs are 4, 9, …, 59 (where `(epoch+1) % 5 == 0`) plus 63, which is
forced as the final epoch so the last model is always eligible for selection.

**Do not tune on the monitoring curve.** It selects `best.pt`; using it to change
the learning rate, spacing, sampler, loss or horizon mid-run would convert it into
a hyperparameter-tuning set and break the matched comparison. The only legitimate
interventions are technical failures: OOM, non-finite loss, or a crash.